# Étape 4 - Nettoyage : Raw -> Silver

Applique les règles de `docs/transformations/regles_nettoyage.md` aux sources
`store_sales`, `item`, `store` et `date_dim` (table des dates, nécessaire pour
traduire les clés `*_date_sk`).

- Entrée : fichiers bruts `.dat` (lus seulement, jamais modifiés). Pour `store_sales`,
  un **échantillon de 10 % des tickets** (`data/samples/store_sales_echantillon.dat`,
  66 593 lignes) : le fichier complet est trop lourd pour la machine. Les tickets sont
  gardés entiers, pour que le nombre de transactions et le panier moyen restent justes.
- Sortie : `data/silver/<table>/` en Parquet.
- Lignes écartées : `data/silver/rejets/<table>/`, avec la colonne `motif_rejet`.

Chaque règle porte un numéro (R1, R2...) repris dans le document de règles.

## 0. Préparation

In [1]:
import json
import os

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

# Racine du projet : on remonte depuis le dossier du notebook
RACINE = os.getcwd()
while os.path.basename(RACINE) != "retail-intelligence-platform":
    RACINE = os.path.dirname(RACINE)
    if RACINE == "/":
        raise RuntimeError("Lancer le notebook depuis le dossier retail-intelligence-platform")

# Sans le prefixe file://, Spark irait chercher les fichiers dans HDFS
LOCAL = "file://"
SILVER = LOCAL + os.path.join(RACINE, "data", "silver")
GOLD = LOCAL + os.path.join(RACINE, "data", "gold")

spark = (
    SparkSession.builder
    .appName("TP3-Nettoyage-Silver")
    .master("local[2]")                            # 2 coeurs : laisse de la place au reste du PC
    .config("spark.sql.shuffle.partitions", "4")   # petit volume : peu de partitions suffisent
    .config("spark.driver.memory", "768m")        # memoire limitee pour ne pas saturer le PC
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

SOURCE = os.path.expanduser("~/DEV/tpcxbb_source/retail_project/data")

# store_sales complet (667 579 lignes) est trop lourd pour ce PC : on travaille sur un
# echantillon de 10 % des tickets (tickets entiers, 66 593 lignes), cree une seule fois avec :
#   awk -F'|' '$10 % 10 == 0' <SOURCE>/store_sales/store_sales.dat > data/samples/store_sales_echantillon.dat
# Pour traiter le fichier complet, remplacer ce chemin par celui de SOURCE.
FICHIERS = {"store_sales": os.path.join(RACINE, "data", "samples", "store_sales_echantillon.dat")}
if not os.path.exists(FICHIERS["store_sales"]):
    raise FileNotFoundError("Creer l'echantillon avec la commande awk ci-dessus")

SCHEMA_SPEC = os.path.join(RACINE, "..", "..", "01_UC01_besoin_exploration", "preuves", "tpcxbb_schema.json")
with open(SCHEMA_SPEC) as f:
    SPEC = json.load(f)

# Bilan des regles : nombre de lignes concernees par chaque regle
bilan = {}

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/18 16:20:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## R1 - Types : schéma explicite

L'inférence de l'étape 3 n'est pas une validation. On impose les types de la
spécification TPCx-BB : `decimal(7,2)` pour les montants (pas d'erreur d'arrondi),
`bigint` pour les clés, `string` pour les codes (`s_zip`, `s_street_number`).

Une valeur qui ne peut pas être convertie est repérée grâce à la colonne
technique `_ligne_corrompue` (mode `PERMISSIVE`) au lieu de devenir un `null` silencieux.

In [2]:
from pyspark.sql.types import (DecimalType, IntegerType, LongType, StringType, StructField,
                               StructType)


def type_spark(t):
    if t == "bigint":
        return LongType()
    if t == "int":
        return IntegerType()
    if t.startswith("decimal"):
        precision, echelle = t[8:-1].split(",")
        return DecimalType(int(precision), int(echelle))
    return StringType()  # string et dates : les dates sont converties a part (R5)


def lire_brut(table):
    """Lit un fichier .dat avec le schema explicite de la specification."""
    schema = StructType(
        [StructField(nom, type_spark(t), True) for nom, t, _ in SPEC[table]]
        + [StructField("_ligne_corrompue", StringType(), True)]
    )
    return (
        spark.read
        .option("header", False)
        .option("sep", "|")
        .option("encoding", "UTF-8")
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_ligne_corrompue")
        .schema(schema)
        .csv(LOCAL + FICHIERS.get(table, os.path.join(SOURCE, table, f"{table}.dat")))
        .cache()  # obligatoire pour filtrer sur la colonne de ligne corrompue
    )

## R2 - Harmonisation des textes

Retirer les espaces inutiles (`trim`) et remplacer les chaînes vides par `null`,
pour qu'une valeur manquante ait toujours la même représentation.

In [3]:
def harmoniser_textes(df):
    for nom, t in df.dtypes:
        if t == "string" and nom != "_ligne_corrompue":
            df = df.withColumn(nom, F.when(F.trim(F.col(nom)) == "", None).otherwise(F.trim(F.col(nom))))
    return df

## Outil commun : séparer les lignes valides des lignes rejetées

Chaque règle de rejet est une condition. Une ligne reçoit le motif de la
**première** règle qu'elle enfreint. Les lignes sans motif sont valides.

In [4]:
def appliquer_rejets(df, regles, table):
    """regles : liste de (motif, condition_de_rejet). Renvoie (valides, rejets)."""
    motif = F.lit(None).cast("string")
    for nom, condition in reversed(regles):
        motif = F.when(condition, F.lit(nom)).otherwise(motif)
    df = df.withColumn("motif_rejet", motif).cache()

    comptes = {r["motif_rejet"]: r["count"] for r in df.groupBy("motif_rejet").count().collect()}
    bilan[table] = {"lignes_lues": df.count(),
                    "rejets_par_motif": {nom: comptes.get(nom, 0) for nom, _ in regles}}

    valides = df.filter(F.col("motif_rejet").isNull()).drop("motif_rejet", "_ligne_corrompue")
    rejets = df.filter(F.col("motif_rejet").isNotNull())
    return valides, rejets


def ecrire(df, chemin):
    df.write.mode("overwrite").parquet(chemin)

## 1. `date_dim` - table des dates

Elle sert de référentiel pour valider et traduire les clés de date.

- R5 : `d_date` est convertie en vrai type `date`, après vérification du format `yyyy-MM-dd`.
- On ne garde que les colonnes utiles à l'analyse.

In [5]:
dates_brut = harmoniser_textes(lire_brut("date_dim"))
dates_brut = dates_brut.withColumn("d_date_conv", F.to_date("d_date", "yyyy-MM-dd"))

dates, rejets_dates = appliquer_rejets(dates_brut, [
    ("R1_type_invalide", F.col("_ligne_corrompue").isNotNull()),
    ("R3_obligatoire_manquant", F.col("d_date_sk").isNull() | F.col("d_date").isNull()),
    ("R5_date_invalide", F.col("d_date_conv").isNull()),
    ("R6_doublon_cle", F.count("*").over(Window.partitionBy("d_date_sk")) > 1),
], "date_dim")

dates = dates.select(
    "d_date_sk", F.col("d_date_conv").alias("d_date"), "d_year", "d_qoy", "d_moy", "d_dom",
    "d_day_name", "d_weekend", "d_holiday",
)
ecrire(dates, SILVER + "/date_dim")
print(bilan["date_dim"])

{'lignes_lues': 109573, 'rejets_par_motif': {'R1_type_invalide': 0, 'R3_obligatoire_manquant': 0, 'R5_date_invalide': 0, 'R6_doublon_cle': 0}}


## 2. `item` - produits

- R3 : `i_item_sk` et `i_item_id` sont obligatoires (clés). Marque et description manquantes sont conservées (`null`).
- R4 : identifiant négatif = rejet.
- R5 : `i_rec_start_date` convertie en `date`.
- R7 : un prix courant nul ou négatif est rejeté. Un coût d'achat supérieur au prix est **conservé** et signalé (`i_cout_superieur_prix`).
- R6 : doublon de clé = rejet (aucun attendu).

In [6]:
produits_brut = harmoniser_textes(lire_brut("item"))
produits_brut = (produits_brut
    .withColumn("i_rec_start_date_conv", F.to_date("i_rec_start_date", "yyyy-MM-dd")))

produits, rejets_produits = appliquer_rejets(produits_brut, [
    ("R1_type_invalide", F.col("_ligne_corrompue").isNotNull()),
    ("R3_obligatoire_manquant", F.col("i_item_sk").isNull() | F.col("i_item_id").isNull()
                                | F.col("i_category").isNull() | F.col("i_current_price").isNull()),
    ("R4_identifiant_invalide", F.col("i_item_sk") < 0),
    ("R5_date_invalide", F.col("i_rec_start_date").isNotNull() & F.col("i_rec_start_date_conv").isNull()),
    ("R7_prix_invalide", F.col("i_current_price") <= 0),
    ("R6_doublon_cle", F.count("*").over(Window.partitionBy("i_item_sk")) > 1),
], "item")

produits = (produits
    .withColumn("i_rec_start_date", F.col("i_rec_start_date_conv"))
    .withColumn("i_rec_end_date", F.to_date("i_rec_end_date", "yyyy-MM-dd"))
    .drop("i_rec_start_date_conv")
    .withColumn("i_cout_superieur_prix", F.col("i_wholesale_cost") > F.col("i_current_price")))

ecrire(produits, SILVER + "/item")
print(bilan["item"])
produits.select("i_item_sk", "i_item_id", "i_product_name", "i_category", "i_current_price",
                "i_wholesale_cost", "i_cout_superieur_prix", "i_rec_start_date").show(5, truncate=False)

{'lignes_lues': 17820, 'rejets_par_motif': {'R1_type_invalide': 0, 'R3_obligatoire_manquant': 0, 'R4_identifiant_invalide': 0, 'R5_date_invalide': 0, 'R7_prix_invalide': 0, 'R6_doublon_cle': 0}}


+---------+----------------+------------------------------------------------+-----------+---------------+----------------+---------------------+----------------+
|i_item_sk|i_item_id       |i_product_name                                  |i_category |i_current_price|i_wholesale_cost|i_cout_superieur_prix|i_rec_start_date|
+---------+----------------+------------------------------------------------+-----------+---------------+----------------+---------------------+----------------+
|2        |AAAAAAAAAAAAAAAC|bMcWnXWH4yWhqADKdlZicQXcIhd48e8iROsScTzHAKPFcgh8|Books      |62.74          |47.70           |false                |2000-09-01      |
|4        |AAAAAAAAAAAAAAAE|WinA                                            |Books      |40.95          |82.68           |true                 |2000-04-04      |
|5        |AAAAAAAAAAAAAAAF|aUJrSEZIUscU8sZqhdKaQGdhYO7BsmUtgHcCZHM9ajAku2  |Movies & TV|26.50          |13.06           |false                |2000-11-18      |
|8        |AAAAAAAAAAAAAAAI|

## 3. `store` - magasins

- R1/R2 : `s_zip` et `s_street_number` restent du texte (`s_zip` complété à 5 chiffres pour garder les zéros initiaux).
- R3, R4, R6 : clé obligatoire, positive, unique.
- R5 : `s_rec_start_date` convertie en `date`. **Date de fermeture** : `s_closed_date_sk` est traduite avec `date_dim`. Une fermeture antérieure à l'ouverture, ou postérieure à la dernière vente connue, est impossible : la date est remplacée par `null` et signalée (`s_date_fermeture_invalide`). Le magasin est **conservé**, car ses ventes sont réelles.

In [7]:
magasins_brut = harmoniser_textes(lire_brut("store"))
magasins_brut = magasins_brut.withColumn("s_rec_start_date_conv", F.to_date("s_rec_start_date", "yyyy-MM-dd"))

magasins, rejets_magasins = appliquer_rejets(magasins_brut, [
    ("R1_type_invalide", F.col("_ligne_corrompue").isNotNull()),
    ("R3_obligatoire_manquant", F.col("s_store_sk").isNull() | F.col("s_store_id").isNull()),
    ("R4_identifiant_invalide", F.col("s_store_sk") < 0),
    ("R5_date_invalide", F.col("s_rec_start_date").isNotNull() & F.col("s_rec_start_date_conv").isNull()),
    ("R6_doublon_cle", F.count("*").over(Window.partitionBy("s_store_sk")) > 1),
], "store")

# Derniere date de vente connue (borne superieure d'une fermeture possible)
derniere_vente = (lire_brut("store_sales").join(dates, F.col("ss_sold_date_sk") == F.col("d_date_sk"))
                  .agg(F.max("d_date")).first()[0])
print("Derniere vente connue :", derniere_vente)

fermeture = dates.select(F.col("d_date_sk").alias("s_closed_date_sk"), F.col("d_date").alias("s_closed_date"))
magasins = (magasins
    .withColumn("s_rec_start_date", F.col("s_rec_start_date_conv")).drop("s_rec_start_date_conv")
    .withColumn("s_rec_end_date", F.to_date("s_rec_end_date", "yyyy-MM-dd"))
    .withColumn("s_zip", F.lpad("s_zip", 5, "0"))
    .join(fermeture, "s_closed_date_sk", "left")
    .withColumn("s_date_fermeture_invalide",
                F.col("s_closed_date").isNotNull()
                & ((F.col("s_closed_date") < F.col("s_rec_start_date"))
                   | (F.col("s_closed_date") > F.lit(derniere_vente))))
    .withColumn("s_closed_date", F.when(F.col("s_date_fermeture_invalide"), None).otherwise(F.col("s_closed_date")))
    .drop("s_closed_date_sk"))

bilan["store"]["dates_fermeture_invalidees"] = magasins.filter("s_date_fermeture_invalide").count()
ecrire(magasins, SILVER + "/store")
print(bilan["store"])
magasins.select("s_store_sk", "s_store_name", "s_rec_start_date", "s_closed_date",
                "s_date_fermeture_invalide", "s_zip", "s_hours").show(truncate=False)

Derniere vente connue : 2005-12-13


{'lignes_lues': 12, 'rejets_par_motif': {'R1_type_invalide': 0, 'R3_obligatoire_manquant': 0, 'R4_identifiant_invalide': 0, 'R5_date_invalide': 0, 'R6_doublon_cle': 0}, 'dates_fermeture_invalidees': 12}


+----------+------------+----------------+-------------+-------------------------+-----+--------+
|s_store_sk|s_store_name|s_rec_start_date|s_closed_date|s_date_fermeture_invalide|s_zip|s_hours |
+----------+------------+----------------+-------------+-------------------------+-----+--------+
|2         |Sherwood    |2000-09-01      |NULL         |true                     |97832|8AM-4PM |
|4         |Elmwood     |2000-07-03      |NULL         |true                     |64295|8AM-4PM |
|5         |Egypt       |2000-03-03      |NULL         |true                     |61142|8AM-4PM |
|8         |Williamstown|2000-06-02      |NULL         |true                     |60921|8AM-4PM |
|0         |Forest      |2000-08-02      |NULL         |true                     |47562|8AM-12AM|
|1         |Hillside    |2000-07-03      |NULL         |true                     |80145|8AM-4PM |
|6         |Bloomfield  |2000-01-03      |NULL         |true                     |43858|8AM-4PM |
|9         |Germanto

## 4. `store_sales` - ventes en magasin

| Règle | Rejet si... |
|---|---|
| R1 | une valeur n'a pas pu être convertie dans son type |
| R3 | une colonne obligatoire est vide : date, produit, magasin, ticket, quantité, prix, montants |
| R4 | un identifiant est négatif, ou le produit / le magasin / la date n'existe pas dans sa table de référence |
| R7 | la quantité est ≤ 0, un prix ou un montant payé est négatif, le prix de vente dépasse le prix catalogue, ou le montant ≠ quantité × prix |
| R6 | la ligne est une copie exacte d'une autre ligne (on garde la première) |

Sont **conservés** :
- les clés client / promotion vides (R3 : information facultative, `ss_promo_sk` vide = vente sans promotion) ;
- les marges négatives et les prix à 0 (R7 : cas métier possibles, signalés par des indicateurs) ;
- les valeurs atypiques (R8 : cohérentes avec quantité × prix) ;
- les couples (ticket, produit) répétés (R6 : lignes différentes, numérotées par `ss_ligne_ticket`).

In [8]:
ventes_brut = lire_brut("store_sales")

OBLIGATOIRES = ["ss_sold_date_sk", "ss_item_sk", "ss_store_sk", "ss_ticket_number", "ss_quantity",
                "ss_list_price", "ss_sales_price", "ss_ext_sales_price", "ss_ext_list_price",
                "ss_ext_discount_amt", "ss_net_paid", "ss_net_profit"]
MONTANTS_POSITIFS = ["ss_wholesale_cost", "ss_list_price", "ss_sales_price", "ss_ext_discount_amt",
                     "ss_ext_sales_price", "ss_ext_wholesale_cost", "ss_ext_list_price", "ss_ext_tax",
                     "ss_coupon_amt", "ss_net_paid", "ss_net_paid_inc_tax"]  # ss_net_profit peut etre negatif

# Cles de reference (une colonne booleenne par table : la cle existe-t-elle ?)
ventes_brut = (ventes_brut
    .join(produits.select(F.col("i_item_sk").alias("ss_item_sk"), F.lit(True).alias("_produit_connu")),
          "ss_item_sk", "left")
    .join(magasins.select(F.col("s_store_sk").alias("ss_store_sk"), F.lit(True).alias("_magasin_connu")),
          "ss_store_sk", "left")
    .join(dates.select(F.col("d_date_sk").alias("ss_sold_date_sk"), F.lit(True).alias("_date_connue")),
          "ss_sold_date_sk", "left"))

def une_colonne(colonnes, test):
    cond = F.lit(False)
    for c in colonnes:
        cond = cond | test(F.col(c))
    return cond

ventes, rejets_ventes = appliquer_rejets(ventes_brut, [
    ("R1_type_invalide", F.col("_ligne_corrompue").isNotNull()),
    ("R3_obligatoire_manquant", une_colonne(OBLIGATOIRES, lambda c: c.isNull())),
    ("R4_identifiant_invalide", une_colonne([c for c, _, _ in SPEC["store_sales"] if c.endswith("_sk")]
                                            + ["ss_ticket_number"], lambda c: c < 0)),
    ("R4_produit_inconnu", F.col("_produit_connu").isNull()),
    ("R4_magasin_inconnu", F.col("_magasin_connu").isNull()),
    ("R4_date_inconnue", F.col("_date_connue").isNull()),
    ("R7_quantite_invalide", F.col("ss_quantity") <= 0),
    ("R7_montant_negatif", une_colonne(MONTANTS_POSITIFS, lambda c: c < 0)),
    ("R7_prix_superieur_catalogue", F.col("ss_sales_price") > F.col("ss_list_price")),
    ("R7_montant_incoherent",
     F.abs(F.col("ss_ext_sales_price") - F.col("ss_quantity") * F.col("ss_sales_price")) > 0.01),
], "store_sales")

ventes = ventes.drop("_produit_connu", "_magasin_connu", "_date_connue")

In [9]:
# R6 - Doublons exacts : suppression (la definition est claire : toutes les colonnes identiques)
avant = ventes.count()
ventes = ventes.dropDuplicates()
bilan["store_sales"]["doublons_exacts_supprimes"] = avant - ventes.count()

# R6 - Couples (ticket, produit) repetes : conserves et numerotes pour obtenir une cle unique
ordre = Window.partitionBy("ss_ticket_number", "ss_item_sk").orderBy(
    "ss_sold_time_sk", "ss_quantity", "ss_sales_price", "ss_net_paid", "ss_promo_sk")
ventes = ventes.withColumn("ss_ligne_ticket", F.row_number().over(ordre))
bilan["store_sales"]["lignes_ticket_produit_repete"] = ventes.filter("ss_ligne_ticket > 1").count()

# R7 - Indicateurs : cas conserves mais signales
ventes = (ventes
    .withColumn("ss_prix_nul", F.col("ss_sales_price") == 0)
    .withColumn("ss_marge_negative", F.col("ss_net_profit") < 0)
    .withColumn("ss_sans_promotion", F.col("ss_promo_sk").isNull()))
for indicateur in ["ss_prix_nul", "ss_marge_negative", "ss_sans_promotion"]:
    bilan["store_sales"][indicateur] = ventes.filter(F.col(indicateur)).count()

ecrire(ventes, SILVER + "/store_sales")
print(json.dumps(bilan["store_sales"], indent=1))

{
 "lignes_lues": 66593,
 "rejets_par_motif": {
  "R1_type_invalide": 0,
  "R3_obligatoire_manquant": 0,
  "R4_identifiant_invalide": 0,
  "R4_produit_inconnu": 0,
  "R4_magasin_inconnu": 0,
  "R4_date_inconnue": 0,
  "R7_quantite_invalide": 0,
  "R7_montant_negatif": 0,
  "R7_prix_superieur_catalogue": 0,
  "R7_montant_incoherent": 0
 },
 "doublons_exacts_supprimes": 0,
 "lignes_ticket_produit_repete": 406,
 "ss_prix_nul": 356,
 "ss_marge_negative": 49559,
 "ss_sans_promotion": 15
}


## 5. Mise à l'écart des enregistrements rejetés

Les lignes rejetées ne sont pas supprimées : elles sont conservées avec leur
motif dans `data/silver/rejets/`, pour pouvoir les analyser ou les signaler à
l'équipe qui a fourni les données.

In [10]:
for nom, rejets in [("date_dim", rejets_dates), ("item", rejets_produits),
                    ("store", rejets_magasins), ("store_sales", rejets_ventes)]:
    rejets = rejets.drop(*[c for c in rejets.columns if c.startswith("_") and c != "_ligne_corrompue"])
    ecrire(rejets, SILVER + f"/rejets/{nom}")
    print(f"{nom:<12} {rejets.count():>6} ligne(s) rejetee(s)")

date_dim          0 ligne(s) rejetee(s)


item              0 ligne(s) rejetee(s)


store             0 ligne(s) rejetee(s)


store_sales       0 ligne(s) rejetee(s)


## 6. Contrôle après nettoyage

On relit les fichiers Silver écrits pour vérifier : nombre de lignes, schéma,
absence de `null` dans les colonnes obligatoires et unicité des clés.

In [11]:
CLES = {"date_dim": ["d_date_sk"], "item": ["i_item_sk"], "store": ["s_store_sk"],
        "store_sales": ["ss_ticket_number", "ss_item_sk", "ss_ligne_ticket"]}

for table, cle in CLES.items():
    df = spark.read.parquet(SILVER + f"/{table}")
    nb = df.count()
    doublons = nb - df.dropDuplicates(cle).count()
    cles_nulles = df.filter(une_colonne(cle, lambda c: c.isNull())).count()
    bilan[table]["lignes_silver"] = nb
    print(f"{table:<12} {nb:>7} lignes | doublons sur {cle} : {doublons} | cles nulles : {cles_nulles}")
    assert doublons == 0 and cles_nulles == 0

spark.read.parquet(SILVER + "/store_sales").printSchema()

date_dim      109573 lignes | doublons sur ['d_date_sk'] : 0 | cles nulles : 0


item           17820 lignes | doublons sur ['i_item_sk'] : 0 | cles nulles : 0


store             12 lignes | doublons sur ['s_store_sk'] : 0 | cles nulles : 0


store_sales    66593 lignes | doublons sur ['ss_ticket_number', 'ss_item_sk', 'ss_ligne_ticket'] : 0 | cles nulles : 0
root
 |-- ss_sold_date_sk: long (nullable = true)
 |-- ss_store_sk: long (nullable = true)
 |-- ss_item_sk: long (nullable = true)
 |-- ss_sold_time_sk: long (nullable = true)
 |-- ss_customer_sk: long (nullable = true)
 |-- ss_cdemo_sk: long (nullable = true)
 |-- ss_hdemo_sk: long (nullable = true)
 |-- ss_addr_sk: long (nullable = true)
 |-- ss_promo_sk: long (nullable = true)
 |-- ss_ticket_number: long (nullable = true)
 |-- ss_quantity: integer (nullable = true)
 |-- ss_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_list_price: decimal(7,2) (nullable = true)
 |-- ss_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_discount_amt: decimal(7,2) (nullable = true)
 |-- ss_ext_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_ext_list_price: decimal(7,2) (nullable = true)
 |-- ss_ext_tax: decima

In [12]:
os.makedirs(os.path.join(RACINE, "outputs"), exist_ok=True)
with open(os.path.join(RACINE, "outputs", "bilan_nettoyage.json"), "w") as f:
    json.dump(bilan, f, indent=1, ensure_ascii=False, default=str)
print(json.dumps(bilan, indent=1, ensure_ascii=False, default=str))

{
 "date_dim": {
  "lignes_lues": 109573,
  "rejets_par_motif": {
   "R1_type_invalide": 0,
   "R3_obligatoire_manquant": 0,
   "R5_date_invalide": 0,
   "R6_doublon_cle": 0
  },
  "lignes_silver": 109573
 },
 "item": {
  "lignes_lues": 17820,
  "rejets_par_motif": {
   "R1_type_invalide": 0,
   "R3_obligatoire_manquant": 0,
   "R4_identifiant_invalide": 0,
   "R5_date_invalide": 0,
   "R7_prix_invalide": 0,
   "R6_doublon_cle": 0
  },
  "lignes_silver": 17820
 },
 "store": {
  "lignes_lues": 12,
  "rejets_par_motif": {
   "R1_type_invalide": 0,
   "R3_obligatoire_manquant": 0,
   "R4_identifiant_invalide": 0,
   "R5_date_invalide": 0,
   "R6_doublon_cle": 0
  },
  "dates_fermeture_invalidees": 12,
  "lignes_silver": 12
 },
 "store_sales": {
  "lignes_lues": 66593,
  "rejets_par_motif": {
   "R1_type_invalide": 0,
   "R3_obligatoire_manquant": 0,
   "R4_identifiant_invalide": 0,
   "R4_produit_inconnu": 0,
   "R4_magasin_inconnu": 0,
   "R4_date_inconnue": 0,
   "R7_quantite_invalide":

In [13]:
spark.stop()